In [70]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import pandas as pd

np.random.seed(42)

In [71]:
S0 = 100
K = 100
r = 0.02
u = 1.2
d = 0.8

n_steps = 3

**American Option Pricing**

The value can be calculated by means of a three, from the final payoff, knowing the price movement one can compare them to the payoff at each step, given that the American Opt give the right to exercise in each moment during the period.

call = max(S_t-K, discouted expectation of next continuation value that in the end is the final payoff, regression method)

In [72]:
# function that is used in the binomial model to price the previous node give the next 2 price (t+1) and the actual price of the stock in t.

# we need to use the risk neutral measure q, found by solving a lineare system given by 2 equations
# (1+r) = q_u * u + q_d * d 
# 1 = q_u + q_d

def succ_el(node,u,d):
    return node*u, node*d

def three_generation(n_step, u=1.2, d=0.8):
    S = 100
    three = [[S]]
    for i in range(1,n_step):
        new_node = []
        for el in three[i-1]:
            new_elements_up, new_elements_down = succ_el(el,u,d)
            if len(new_node) == 0:
                new_node += [new_elements_up,new_elements_down]
            else:    
                new_node += [new_elements_down]
        three.append(new_node)

    return three
        

In [73]:
# we need to go from the price three to the payoff three

three = three_generation(n_steps+1)

payoff_three = [row[:] for row in three]
payoff_three[-1] = [max(S-K,0) for S in three[-1]]

In [74]:
q_u = ((1+r) - d)/(u-d)
q_d = 1-q_u


for i in range(len(three)-1,0,-1):
    l = []
    for ind,el in enumerate(three[i-1]):
        continuation = (1/(1+r)) * (q_u * payoff_three[i][ind] + q_d * payoff_three[i][ind+1])
        early_exercise = max(el-K, 0)
        l.append(max(early_exercise,continuation))
    payoff_three[i-1] = l

print("American Call Price in t=0: ", round(payoff_three[0][0],3))

American Call Price in t=0:  17.263
